# Notebook overview: R8_3-CO_Randomgraph.ipynb

## What this notebook does
Builds and evaluates random-survival counterfactual graphs from previously sampled survivor lists. The notebook loads observed pre/post graphs, induces graphs on sampled survivors, computes network-level metrics, and saves counterfactual graph ensembles.

## Inputs used
- Observed pre-disaster graph pickle: PDM_Oct2021_graph_POI_weighted_{revision}.pkl
- Observed post-disaster remaining-pre-users graph pickle: Jan2022_remaining_pre_users_graph_POI_weighted_{revision}.pkl
- Survivor JSON files: graphs_POI_weighted/boots_user_survivors/{revision}/survivors_run*.json
- Python helper: build_monthly_graphs_universal.py when graph rebuilding is needed

## Outputs created
- Counterfactual graph pickles: CF_graph_run*_*.pkl
- Network metric CSVs for observed and random-survival graphs
- Centrality/summary outputs used by downstream comparison notebooks

**Data access warning.** The Cuebiq/Spectus mobility stop data used by this project are proprietary/restricted and are not included in this repository. Do not commit raw mobility files, user IDs, stop tables, home-location files, graph outputs containing sensitive identifiers, or large derived files unless your data-use agreement explicitly permits release. Public users must obtain data access independently and place files in the documented paths.

In [ ]:
pip install statsmodels 

In [ ]:
pip install seaborn

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import numpy as np
from scipy.sparse import lil_matrix
import json
from collections import defaultdict
import networkx as nx
import pickle
import os
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
import seaborn as sns

In [ ]:
base = os.path.join("..", "Data", "stop_df_perday_CO")
pois_dir = os.path.join(base, "POIs")
geo_dir = os.path.join(base, "geography_CO")
stops_dir = os.path.join(base, "daily_agg_to_weekly_Stops")
graph_poi_dir = os.path.join(base, "graphs", "poi-user")
os.makedirs(graph_poi_dir, exist_ok=True)
graph_dir = os.path.join(base, "graphs_POI_weighted")
os.makedirs(graph_dir, exist_ok=True)
home_dir = os.path.join(base, "home/pre_disaster")

In [ ]:
revision = "R11"

survivor_dir = os.path.join(graph_poi_dir, "boots_user_survivors", revision)
os.makedirs(survivor_dir, exist_ok=True)

In [ ]:
import sys
import os

sys.path.append(os.getcwd())

# Load Pre and post graphs

In [ ]:
from build_monthly_graphs_universal import build_monthly_graphs

In [ ]:
import sys
sys.path.append("/path/to/your/script_folder")

from build_monthly_graphs_universal import build_monthly_graphs

In [ ]:
pre_graph_path = os.path.join(
    graph_dir,
    f"PDM_Oct2021_graph_POI_weighted_{revision}.pkl"
)

with open(pre_graph_path, "rb") as f:
    G_pre = pickle.load(f)

print("Nodes:", G_pre.number_of_nodes())
print("Edges:", G_pre.number_of_edges())

In [ ]:
post_graph_path = os.path.join(
    graph_dir,
    f"Jan2022_remaining_pre_users_graph_POI_weighted_{revision}.pkl"
)

with open(post_graph_path, "rb") as f:
    G_post = pickle.load(f)

print("Nodes:", G_post.number_of_nodes())
print("Edges:", G_post.number_of_edges())

In [ ]:
print(type(next(iter(G_pre.nodes()))))

# Building Random Graphs

In [ ]:
survivor_files = sorted(
    [f for f in os.listdir(survivor_dir) if f.startswith("survivors_run")]
)

In [ ]:
cf_graphs = {}
cf_metrics = []

for f in survivor_files:

    run = int(f.split("run")[1].split(".")[0])

    with open(os.path.join(survivor_dir, f)) as fp:
        survivors = set(map(int, json.load(fp)))  # must match graph node type

    # Induced subgraph = remove all non-survivor nodes
    G_cf = G_pre.subgraph(survivors).copy()

    cf_graphs[run] = G_cf

    # Collect basic network stats immediately
    nodes = G_cf.number_of_nodes()
    edges = G_cf.number_of_edges()

    mean_strength = (
        sum(dict(G_cf.degree(weight="weight")).values()) / nodes
        if nodes > 0 else 0
    )

    cf_metrics.append({
        "run": run,
        "nodes": nodes,
        "edges": edges,
        "mean_strength": mean_strength,
        "type": "random_post"
    })

    print(
        f"Run {run:04d} | Nodes={nodes} | "
        f"Edges={edges} | Mean strength={mean_strength:.3f}"
    )

# Computing node level metrics

In [ ]:
# Ensure directory exists
os.makedirs(survivor_dir, exist_ok=True)

cf_metrics = []

for f in survivor_files:

    run = int(f.split("run")[1].split(".")[0])

    with open(os.path.join(survivor_dir, f)) as fp:
        survivors = set(map(int, json.load(fp)))  # match node type

    # Induced subgraph
    G_cf = G_pre.subgraph(survivors).copy()

    # -----------------------------
    # Save graph (.pkl)
    # -----------------------------
    graph_path = os.path.join(
        survivor_dir,
        f"CF_graph_run{run:04d}_Oct2021_R11.pkl"
    )

    with open(graph_path, "wb") as gf:
        pickle.dump(G_cf, gf)

    # -----------------------------
    # Compute metrics
    # -----------------------------
    nodes = G_cf.number_of_nodes()
    edges = G_cf.number_of_edges()

    mean_strength = (
        sum(dict(G_cf.degree(weight="weight")).values()) / nodes
        if nodes > 0 else 0
    )

    clustering = (
        nx.average_clustering(G_cf, weight="weight")
        if nodes > 0 else 0
    )

    cf_metrics.append({
        "run": run,
        "nodes": nodes,
        "edges": edges,
        "mean_strength": mean_strength,
        "clustering": clustering
    })

    print(
        f"Run {run:04d} | Nodes={nodes} | "
        f"Edges={edges} | Mean strength={mean_strength:.3f}"
    )

# ----------------------------------
# Save metrics CSV
# ----------------------------------
cf_metrics_df = pd.DataFrame(cf_metrics)

metrics_path = os.path.join(
    survivor_dir,
    "CF_network_metrics_Oct2021_{revision}.csv"
)

cf_metrics_df.to_csv(metrics_path, index=False)

print("Saved CF metrics →", metrics_path)

# Plotting metrics

In [ ]:
random_network_path = os.path.join(
    survivor_dir,
    f"CF_network_metrics_Oct2021_{revision}.csv"
)

random_network_df = pd.read_csv(random_network_path)

In [ ]:
random_network_df.columns

In [ ]:
graph_dir = "../Data/stop_df_perday_CO/graphs_POI_weighted"
network_path = os.path.join(
    graph_dir,
    "network_centrality_POI_weighted_all_revisions.csv"
)

network_df = pd.read_csv(network_path)


In [ ]:
# Filter R11 only
r11_df = network_df[network_df["revision"] == "R11"].copy()

# Keep only required columns
r11_structural = r11_df[[
    "month",
    "phase",
    "nodes",
    "edges",
    "mean_strength"
]].copy()


# Add run + type columns to match random_network_df
r11_structural["run"] = -1
r11_structural["type"] = r11_structural["phase"].apply(
    lambda x: f"observed_{x}"
)

r11_structural

In [ ]:
random_structural = random_network_df[[
    "nodes",
    "edges",
    "mean_strength",
    "run"
]].copy()

random_structural["month"] = "random_post"
random_structural["phase"] = "random"

In [ ]:
combined_df = pd.concat(
    [random_structural, r11_structural],
    ignore_index=True
)

In [ ]:
combined_df

In [ ]:
combined_df_network_path = os.path.join(
    survivor_dir,
    f"all_network_metrics_Oct2021_{revision}.csv"
)

combined_df.to_csv(combined_df_network_path, index = False)

In [ ]:
plot_metrics = ["nodes", "edges", "mean_strength"]

palette = {
    "observed_pre": "skyblue",
    "observed_post": "salmon",
    "random_post": "lightgreen"
}

fig, axes = plt.subplots(
    1, len(plot_metrics),
    figsize=(8, 3),
    sharey=False
)

for ax, metric in zip(axes, plot_metrics):

    # ----------------------------
    # 1️⃣ Random distribution
    # ----------------------------
    rand_vals = combined_df.loc[
        combined_df["month"] == "random_post",
        metric
    ]

    rand_mean = rand_vals.mean()
    rand_std  = rand_vals.std()

    # 95% CI
    rand_ci = 1.96 * rand_std

    # ----------------------------
    # 2️⃣ Observed values
    # ----------------------------
    obs_pre = combined_df.loc[
        combined_df["type"] == "observed_pre",
        metric
    ].mean()

    obs_post = combined_df.loc[
        combined_df["type"] == "observed_post",
        metric
    ].mean()

    # ----------------------------
    # 3️⃣ Plot
    # ----------------------------
    labels = ["Observed Pre", "Observed Post", "Random Post"]
    x = np.arange(len(labels))

    values = [obs_pre, obs_post, rand_mean]
    colors = [
        palette["observed_pre"],
        palette["observed_post"],
        palette["random_post"]
    ]

    ax.bar(x, values, color=colors, width=0.6)

    # Add CI errorbar only to random
    ax.errorbar(
        x[2],
        rand_mean,
        yerr=rand_ci,
        fmt="none",
        color="black",
        capsize=5,
        linewidth=1
    )

    ax.set_xticks(x)
    ax.set_xticklabels(["Pre", "Post", "Random"])
    ax.set_title(metric.replace("_", " ").title())

    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.show()

# Computing Centralities

In [ ]:
def ensure_str_nodes(G):
    if any(not isinstance(n, str) for n in G.nodes()):
        return nx.relabel_nodes(G, {n: str(n) for n in G.nodes()}, copy=True)
    return G

In [ ]:
def compute_node_centralities_contact_fast(G, weight="weight", approx_betweenness_k=500, seed=0):
    # build inv_weight without mutating original edge dicts too much
    inv = {}
    for u, v, d in G.edges(data=True):
        w = d.get(weight, 0.0)
        inv[(u, v)] = (1.0 / w) if (w is not None and w > 0) else 0.0
    nx.set_edge_attributes(G, inv, "inv_weight")

    out = pd.DataFrame(index=list(G.nodes()))
    out["raw_degree"] = pd.Series(dict(G.degree()))
    out["strength"] = pd.Series(dict(G.degree(weight=weight)))

    # closeness using inv_weight as distance
    out["closeness_centrality"] = pd.Series(nx.closeness_centrality(G, distance="inv_weight"))

    # clustering (unweighted, unless you want weighted clustering separately)
    out["clustering_coefficient"] = pd.Series(nx.clustering(G))

    # # betweenness: approximate for speed
    # if approx_betweenness_k is None:
    #     out["betweenness_centrality"] = pd.Series(nx.betweenness_centrality(G))
    # else:
    #     out["betweenness_centrality"] = pd.Series(
    #         nx.betweenness_centrality(G, k=approx_betweenness_k, seed=seed)
    #     )

    return out

In [ ]:
cf_graph_files = sorted(
    [f for f in os.listdir(boot_dir_w)
     if f.startswith("CF_graph_run") and f.endswith(".pkl")]
)

cf_node_rows = []
cf_network_rows = []

for f in cf_graph_files:

    run = int(f.split("run")[1].split("_")[0])
    graph_path = os.path.join(survivor_dir, f)

    with open(graph_path, "rb") as fp:
        G = pickle.load(fp)

    nodes = G.number_of_nodes()
    edges = G.number_of_edges()

    print(f"\n🔁 Run {run:04d}")
    print(f"   Nodes: {nodes}")
    print(f"   Edges: {edges}")

    if nodes == 0:
        print("   ⚠️ Empty graph — skipping.")
        continue

    # Ensure consistent node type
    G = ensure_str_nodes(G)

    # -----------------------------
    # Node-level centralities
    # -----------------------------
    node_cent = compute_node_centralities_contact_fast(
        G,
        weight="weight",
        approx_betweenness_k=500,
        seed=0
    )

    print("   ✅ Node centralities computed")

    # Quick sanity stats
    print(f"      Mean strength: {node_cent['strength'].mean():.3f}")
    print(f"      Mean clustering: {node_cent['clustering_coefficient'].mean():.3f}")

    node_cent = node_cent.reset_index().rename(columns={"index": "node"})
    node_cent["run"] = run
    node_cent["phase"] = "random_post"
    node_cent["revision"] = revision

    cf_node_rows.append(node_cent)

    # -----------------------------
    # Network-level summary
    # -----------------------------
    row = {
        "run": run,
        "revision": revision,
        "phase": "random_post",
        "nodes": nodes,
        "edges": edges,

        "mean_strength": float(node_cent["strength"].mean()),
        "median_strength": float(node_cent["strength"].median()),
        "mean_closeness": float(node_cent["closeness_centrality"].mean()),
        "median_closeness": float(node_cent["closeness_centrality"].median()),
        # "mean_betweenness": float(node_cent["betweenness_centrality"].mean()),
        # "median_betweenness": float(node_cent["betweenness_centrality"].median()),
        "mean_clustering": float(node_cent["clustering_coefficient"].mean()),
        "median_clustering": float(node_cent["clustering_coefficient"].median()),
    }

    cf_network_rows.append(row)

    print("   ✅ Network metrics aggregated")

In [ ]:
cf_nodes_df = pd.concat(cf_node_rows, ignore_index=True) if len(cf_node_rows) else pd.DataFrame()
cf_network_df = pd.DataFrame(cf_network_rows) if len(cf_network_rows) else pd.DataFrame()

nodes_csv = os.path.join(survivor_dir, f"cf_node_metrics_{revision}.csv")
net_csv   = os.path.join(survivor_dir, f"cf_network_metrics_{revision}.csv")

cf_nodes_df.to_csv(nodes_csv, index=False)
cf_network_df.to_csv(net_csv, index=False)

print("Saved:")
print(" -", nodes_csv)
print(" -", net_csv)

In [ ]:
cf_nodes_df

# combine with pre and post Centralities

In [ ]:
real_network_df = pd.read_csv(
    "../Data/stop_df_perday_CO/graphs_POI_weighted/network_centrality_POI_weighted_R11.csv"
)

cf_network_df = pd.read_csv(
    os.path.join(survivor_dir, f"cf_network_metrics_{revision}.csv")
)

cf_network_df["month"] = "Oct2021"
cf_network_df["phase"] = "random_post"

common_cols = [
    "revision",
    "month",
    "phase",
    "nodes",
    "edges",
    "mean_strength",
    "median_strength",
    "mean_closeness",
    "median_closeness",
    # "mean_betweenness",
    # "median_betweenness",
    "mean_clustering",
    "median_clustering",
]

combined_network_df = pd.concat(
    [
        real_network_df[common_cols],
        cf_network_df[common_cols],
    ],
    ignore_index=True
)

print(combined_network_df["phase"].value_counts())

In [ ]:
real_node_df = pd.read_csv(
    "../Data/stop_df_perday_CO/graphs_POI_weighted/node_metrics_centrality_POI_weighted_R11.csv"
)

cf_node_df = pd.read_csv(
    os.path.join(survivor_dir, f"cf_node_metrics_{revision}.csv")
)

# ensure these exist for concat
real_node_df["month"] = "Oct2021"
cf_node_df["month"] = "Oct2021"

# keep your intended labels
cf_node_df["phase"] = "random_post"   # real_node_df keeps its existing phase (or set it explicitly if you want)

# ---- NEW: ensure 'run' exists in both ----
if "run" not in real_node_df.columns:
    real_node_df["run"] = "real"   # or -1
# (cf_node_df should already have run; if not, fill)
if "run" not in cf_node_df.columns:
    cf_node_df["run"] = np.nan
# -----------------------------------------

common_node_cols = [
    "node",
    "raw_degree",
    "strength",
    "closeness_centrality",
    "clustering_coefficient",
    "run",         # <--- include run
    "month",
    "phase",
    "revision",
]

# include "num" if present in either df
if ("num" in real_node_df.columns) or ("num" in cf_node_df.columns):
    if "num" not in real_node_df.columns:
        real_node_df["num"] = np.nan
    if "num" not in cf_node_df.columns:
        cf_node_df["num"] = np.nan

    # insert num right after run (or wherever you prefer)
    common_node_cols.insert(common_node_cols.index("month"), "num")

combined_node_df = pd.concat(
    [
        real_node_df[common_node_cols],
        cf_node_df[common_node_cols],
    ],
    ignore_index=True
)

print(combined_node_df["phase"].value_counts())
print("Columns:", combined_node_df.columns.tolist())
print("Run values (sample):", combined_node_df["run"].dropna().unique()[:10])

In [ ]:
csv_path = os.path.join(survivor_dir, f"combined_node_centralities_{revision}.csv")
combined_node_df.to_csv(csv_path, index=False)
print("Saved:")
print(" -", csv_path)

# Plotting Centralities

In [ ]:
csv_path = os.path.join(survivor_dir, f"combined_node_centralities_{revision}.csv")
df = pd.read_csv(csv_path)

# Plotting Graphs:

In [ ]:
def load_nx_graph(pkl_path):
    with open(pkl_path, "rb") as f:
        G = pickle.load(f)
    if not isinstance(G, (nx.Graph, nx.DiGraph, nx.MultiGraph, nx.MultiDiGraph)):
        raise TypeError(f"Loaded object is not a NetworkX graph: {type(G)} from {pkl_path}")
    return G

In [ ]:
# Load random graph:
random_run_id = 11

cf_graph_path = os.path.join(
    survivor_dir,
    f"CF_graph_run{random_run_id:04d}_Oct2021_R11.pkl"
)

G_random = load_nx_graph(cf_graph_path)

In [ ]:
def fragmentation_stats(G):
    comps = list(nx.connected_components(G))
    sizes = [len(c) for c in comps]

    lcc_size = max(sizes)
    n_components = len(sizes)

    print("Total nodes:", G.number_of_nodes())
    print("Number of components:", n_components)
    print("Largest component size:", lcc_size)
    print("Fraction in LCC:", lcc_size / G.number_of_nodes())

In [ ]:
fragmentation_stats(G_pre)
fragmentation_stats(G_post)
fragmentation_stats(G_random)

In [ ]:
def normalize_random_to_post_rules(G_cf):
    """
    Make a Random CF graph comparable to Post graphs built by build_monthly_graphs():
    - Keep only nodes that participate in >=1 edge
    """
    nodes_with_edges = [n for n, d in G_cf.degree() if d > 0]
    return G_cf.subgraph(nodes_with_edges).copy()

In [ ]:
random_run_id = 1

cf_graph_path = os.path.join(
    survivor_dir,
    f"CF_graph_run{random_run_id:04d}_Oct2021_R11.pkl"
)
G_cf = load_nx_graph(cf_graph_path)

G_cf_norm = normalize_random_to_post_rules(G_cf)

print("Original CF:", G_cf.number_of_nodes(), G_cf.number_of_edges())
print("Normalized CF:", G_cf_norm.number_of_nodes(), G_cf_norm.number_of_edges())

In [ ]:
boot_dir_w = os.path.join(graph_poi_dir, "boots_user_removal_weighted_yhat", revision)
os.makedirs(boot_dir_w, exist_ok=True)

In [ ]:
run = 1

graph_path = os.path.join(
        boot_dir_w,
        f"CF_graph_run{run:04d}_Oct2021_{revision}.pkl"
)
G_cf2 = load_nx_graph(graph_path)

G_cf2_norm = normalize_random_to_post_rules(G_cf2)

print("Original CF:", G_cf2.number_of_nodes(), G_cf2.number_of_edges())
print("Normalized CF:", G_cf2_norm.number_of_nodes(), G_cf2_norm.number_of_edges())

## Kamada kawai

In [ ]:
def plot_pre_post_random_kamada(
    G_pre,
    G_post,
    G_rand,
    weight="weight",
    node_size=4,
    node_alpha=0.5,
    edge_alpha=0.15,
    max_edge_width=3.0,
    figsize=(18, 6),
):

    G_union = nx.compose_all([G_pre, G_post, G_rand])
    pos = nx.kamada_kawai_layout(G_union)
    
    all_weights = []

    for G in [G_pre, G_post, G_rand]:
        for _, _, d in G.edges(data=True):
            w = float(d.get(weight, 1.0))
            if w > 0:
                all_weights.append(w)

    global_max = max(all_weights) if all_weights else 1.0

    def scaled_widths(G):
        return [
            max_edge_width * (float(d.get(weight, 1.0)) / global_max)
            for _, _, d in G.edges(data=True)
        ]


    fig, axes = plt.subplots(1, 3, figsize=figsize)

    graphs = [
        (G_pre,  "Pre",  "skyblue"),
        (G_post, "Post", "salmon"),
        (G_rand, "Random CF", "gray"),
    ]

    for ax, (G, title, color) in zip(axes, graphs):

        nx.draw_networkx_nodes(
            G, pos,
            ax=ax,
            node_size=node_size,
            node_color=color,
            alpha=node_alpha,
            linewidths=0
        )

        nx.draw_networkx_edges(
            G, pos,
            ax=ax,
            width=scaled_widths(G),
            edge_color="black",
            alpha=edge_alpha
        )

        ax.set_title(f"{title}\n|V|={G.number_of_nodes()}  |E|={G.number_of_edges()}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# plot_pre_post_random_kamada(
#     G_pre,
#     G_post,
#     G_cf_norm, 
#     weight="weight"
# )

In [ ]:
print(type(next(iter(G_pre.nodes()))))

In [ ]:
print(type(next(iter(G_cf_norm.nodes()))))

## Kamada Kawai

In [ ]:
def plot_pre_post_random_kamada_fast(
    G_pre,
    G_post,
    G_rand,
    weight="weight",
    node_size=4,
    node_alpha=0.5,
    edge_alpha=0.15,
    max_edge_width=3.0,
    figsize=(18, 6),
    seed=0,
    jitter_scale=0.01,
):

    rng = np.random.default_rng(seed)

    # --- Layout only on PRE (much faster)
    pos_pre = nx.spring_layout(G_pre, iterations=30, seed=0)

    # Center for new nodes
    center = np.mean(np.array(list(pos_pre.values())), axis=0)

    def build_positions(G):
        pos = {}
        for n in G.nodes():
            if n in pos_pre:
                pos[n] = pos_pre[n]
            else:
                pos[n] = center + rng.normal(scale=jitter_scale, size=2)
        return pos

    # --- Global weight scaling
    all_weights = [
        float(d.get(weight, 1.0))
        for G in [G_pre, G_post, G_rand]
        for _, _, d in G.edges(data=True)
        if float(d.get(weight, 1.0)) > 0
    ]

    global_max = max(all_weights) if all_weights else 1.0

    def scaled_widths(G):
        return [
            max_edge_width * (float(d.get(weight, 1.0)) / global_max)
            for _, _, d in G.edges(data=True)
        ]

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    graphs = [
        (G_pre,  "Pre",  "skyblue"),
        (G_post, "Post", "salmon"),
        (G_rand, "Random CF", "gray"),
    ]

    for ax, (G, title, color) in zip(axes, graphs):

        pos = build_positions(G)

        nx.draw_networkx_nodes(
            G, pos,
            ax=ax,
            node_size=node_size,
            node_color=color,
            alpha=node_alpha,
            linewidths=0
        )

        nx.draw_networkx_edges(
            G, pos,
            ax=ax,
            width=scaled_widths(G),
            edge_color="black",
            alpha=edge_alpha
        )

        ax.set_title(f"{title}\n|V|={G.number_of_nodes()}  |E|={G.number_of_edges()}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_pre_post_random_kamada_fast(
    G_pre,
    G_post,
    G_cf_norm, 
    weight="weight"
)

### Ignore following

In [ ]:
def largest_component(G):
    if G.number_of_nodes() == 0:
        return G.copy()
    H = G.to_undirected()
    comp = max(nx.connected_components(H), key=len)
    return G.subgraph(comp).copy()

def fast_layout(G, seed=0, method="spring", k=None, iterations=40):
    if method == "spectral":
        return nx.spectral_layout(G)
    if method == "kamada_kawai":
        return nx.kamada_kawai_layout(G)
    return nx.spring_layout(G, seed=seed, k=k, iterations=iterations)

def edge_widths_from_attr(G, weight="weight", min_w=0.3, max_w=4.0, transform="log1p"):
    w = []
    for _, _, d in G.edges(data=True):
        val = d.get(weight, 1.0)
        if val is None or (isinstance(val, float) and np.isnan(val)):
            val = 1.0
        w.append(float(val))

    w = np.asarray(w, dtype=float)
    if w.size == 0:
        return []

    if transform == "log1p":
        w_t = np.log1p(np.maximum(w, 0))
    elif transform == "sqrt":
        w_t = np.sqrt(np.maximum(w, 0))
    else:
        w_t = w.copy()

    w_min, w_max = np.nanmin(w_t), np.nanmax(w_t)
    if not np.isfinite(w_min) or not np.isfinite(w_max) or np.isclose(w_min, w_max):
        return [0.8] * len(w)

    scaled = (w_t - w_min) / (w_max - w_min)
    return (min_w + scaled * (max_w - min_w)).tolist()

def plot_pre_post_no_intersection(
    G_pre,
    G_post,
    *,
    weight="weight",
    layout_method="spring",
    seed=0,
    iterations=40,
    node_size=8,
    node_alpha=0.9,
    edge_alpha=0.20,
    min_edge_width=0.2,
    max_edge_width=5.0,
    width_transform="log1p",
    figsize=(16, 7),
    title_pre="Pre",
    title_post="Post",
    place_new_post_nodes="jitter",  # {"center","jitter"}
    jitter_scale=1e-3,
):
    # take largest component separately (so both are non-empty & readable)
    Gp = largest_component(G_pre)
    Gq = largest_component(G_post)

    # layout on PRE
    pos_pre = fast_layout(Gp, seed=seed, method=layout_method, iterations=iterations)

    # build POST positions using PRE positions where possible
    rng = np.random.default_rng(seed)
    pos_post = {}
    if len(pos_pre) > 0:
        center = np.mean(np.array(list(pos_pre.values())), axis=0)
    else:
        center = np.array([0.0, 0.0])

    for n in Gq.nodes():
        if n in pos_pre:
            pos_post[n] = pos_pre[n]
        else:
            if place_new_post_nodes == "center":
                pos_post[n] = center
            else:  # jitter
                pos_post[n] = center + rng.normal(scale=jitter_scale, size=2)

    # widths
    pre_w  = edge_widths_from_attr(Gp, weight=weight, min_w=min_edge_width, max_w=max_edge_width, transform=width_transform)
    post_w = edge_widths_from_attr(Gq, weight=weight, min_w=min_edge_width, max_w=max_edge_width, transform=width_transform)

    fig, axes = plt.subplots(1, 2, figsize=figsize, constrained_layout=True)

    ax = axes[0]
    nx.draw_networkx_nodes(
    Gp, pos_pre, ax=ax,
    node_size=4,            # smaller
    node_color="skyblue",      # grey
    alpha=0.25,             # translucent
    linewidths=0            # no borders
)

    nx.draw_networkx_edges(Gp, pos_pre, ax=ax, width=pre_w, alpha=edge_alpha)
    ax.set_title(f"{title_pre}\n|V|={Gp.number_of_nodes()}, |E|={Gp.number_of_edges()}\nedge width: '{weight}'")
    ax.axis("off")

    ax = axes[1]
    nx.draw_networkx_nodes(
    Gq, pos_post, ax=ax,
    node_size=4,
    node_color="lightgreen",
    alpha=0.5,
    linewidths=0
)
    nx.draw_networkx_edges(Gq, pos_post, ax=ax, width=post_w, alpha=edge_alpha)
    ax.set_title(f"{title_post}\n|V|={Gq.number_of_nodes()}, |E|={Gq.number_of_edges()}\nedge width: '{weight}', font_size = 14")
    ax.axis("off")

    plt.show()

In [ ]:
pre_graphs = {m: load_nx_graph(p) for m, p in pre_graph_files.items()}

post_remaining_graphs = {}
for month in post_months:
    post_remain_path = os.path.join(graph_dir, f"{month}_remaining_pre_users_graph_POI_weighted_{revision}.pkl")
    post_remaining_graphs[month] = load_nx_graph(post_remain_path)

# --- run ---
plot_pre_post_no_intersection(
    pre_graphs["Oct2021"],
    post_remaining_graphs["Jan2022"],
    weight="weight",                 # or "composite_weight"
    layout_method="spring",
    iterations=40,
    width_transform="log1p",
    title_pre="Pre: Oct2021",
    title_post="Post: Jan2022 (remaining pre users)",
)

In [ ]:
def plot_pre_real_random(
    G_pre,
    G_post_real,
    G_random,
    *,
    weight="weight",
    layout_method="spring",
    seed=0,
    iterations=40,
    node_size=4,
    edge_alpha=0.10,
    # min_edge_width=0.5,
    # max_edge_width=5.0,
    width_transform="log1p",
    figsize=(20, 6),
    jitter_scale=1e-3,
):
    import numpy as np
    import matplotlib.pyplot as plt
    import networkx as nx

    rng = np.random.default_rng(seed)

    # -------------------------------------------------------
    # 1️⃣ Layout based ONLY on PRE largest component
    # -------------------------------------------------------
    if G_pre.number_of_nodes() == 0:
        raise ValueError("Pre graph empty")

    pre_lcc_nodes = max(nx.connected_components(G_pre), key=len)
    G_pre_lcc = G_pre.subgraph(pre_lcc_nodes)

    if layout_method == "spectral":
        pos_pre = nx.spectral_layout(G_pre_lcc)
    elif layout_method == "kamada_kawai":
        pos_pre = nx.kamada_kawai_layout(G_pre_lcc)
    else:
        pos_pre = nx.spring_layout(G_pre_lcc, seed=seed, iterations=iterations)

    center = np.mean(np.array(list(pos_pre.values())), axis=0)

    # -------------------------------------------------------
    # 2️⃣ Position helper
    # -------------------------------------------------------
    def build_positions(G):
        pos = {}
        for n in G.nodes():
            if n in pos_pre:
                pos[n] = pos_pre[n]
            else:
                pos[n] = center + rng.normal(scale=jitter_scale, size=2)
        return pos

    pos_pre_full = build_positions(G_pre)
    pos_post = build_positions(G_post_real)
    pos_rand = build_positions(G_random)

    # -------------------------------------------------------
    # 3️⃣ Edge width scaling
    # -------------------------------------------------------
    def edge_widths(G):
        w = []
        for _, _, d in G.edges(data=True):
            val = d.get(weight, 1.0)
            w.append(float(val))

        w = np.asarray(w)
        if len(w) == 0:
            return []

        if width_transform == "log1p":
            w = np.log1p(np.maximum(w, 0))
        elif width_transform == "sqrt":
            w = np.sqrt(np.maximum(w, 0))

        w_min, w_max = w.min(), w.max()
        if np.isclose(w_min, w_max):
            return [0.8] * len(w)

        scaled = (w - w_min) / (w_max - w_min)
        return w# (min_edge_width + scaled * (max_edge_width - min_edge_width)).tolist()

    # -------------------------------------------------------
    # 4️⃣ Plot
    # -------------------------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=figsize, constrained_layout=True)

    graphs = [
        (G_pre, pos_pre_full, "Pre"),
        (G_post_real, pos_post, "Real Post (Remaining Pre Users)"),
        (G_random, pos_rand, "Random Survival-Only CF")
    ]

    colors = ["gray", "gray", "gray"]

    for ax, (G, pos, title), color in zip(axes, graphs, colors):

        nx.draw_networkx_nodes(
            G, pos, ax=ax,
            node_size=node_size,
            node_color=color,
            alpha=0.5,
            linewidths=0
        )

        nx.draw_networkx_edges(
            G, pos, ax=ax,
            width=edge_widths(G),
            alpha=edge_alpha
        )

        ax.set_title(
            f"{title}\n|V|={G.number_of_nodes()}, |E|={G.number_of_edges()}",
            fontsize=13
        )
        ax.axis("off")

    plt.show()

In [ ]:


plot_pre_real_random(
    pre_graphs["Oct2021"],
    post_remaining_graphs["Jan2022"],
    G_random,
    weight="weight",
    layout_method="spring",
    iterations=40
)

In [ ]:
def plot_pre_post_random_simple(
    G_pre, G_post, G_rand,
    *,
    weight="weight",
    seed=0,
    iterations=40,
    node_size=2,
    node_alpha=0.35,
    edge_alpha=0.08,
    base_edge_width=0.8,
    figsize=(20, 6),
    jitter_scale=1e-3,
):

    rng = np.random.default_rng(seed)

    if G_pre.number_of_nodes() == 0:
        raise ValueError("Pre graph empty")

    # --- LCC of PRE for layout anchoring ---
    H = G_pre.to_undirected()
    lcc = max(nx.connected_components(H), key=len)
    G_pre_lcc = G_pre.subgraph(lcc).copy()

    pos_pre = nx.spring_layout(G_pre_lcc, seed=seed, iterations=iterations)
    center = np.mean(np.array(list(pos_pre.values())), axis=0)

    def remove_isolates(G):
        """Return graph copy without degree-0 nodes."""
        nodes_with_edges = [n for n, d in G.degree() if d > 0]
        return G.subgraph(nodes_with_edges).copy()

    def build_positions(G):
        pos = {}
        for n in G.nodes():
            if n in pos_pre:
                pos[n] = pos_pre[n]
            else:
                pos[n] = center + rng.normal(scale=jitter_scale, size=2)
        return pos

    def edge_widths(G):
        w = np.array(
            [float(d.get(weight, 1.0)) for _, _, d in G.edges(data=True)],
            dtype=float
        )
        if w.size == 0:
            return []
        w = np.maximum(w, 0.0)
        widths = base_edge_width * w
        return widths.tolist()

    # --- Remove isolates ONLY for plotting ---
    G_pre_plot  = remove_isolates(G_pre)
    G_post_plot = remove_isolates(G_post)
    G_rand_plot = remove_isolates(G_rand)

    pos_pre_full = build_positions(G_pre_plot)
    pos_post = build_positions(G_post_plot)
    pos_rand = build_positions(G_rand_plot)

    fig, axes = plt.subplots(1, 3, figsize=figsize, constrained_layout=True)

    panels = [
        (G_pre_plot,  pos_pre_full, "Pre"),
        (G_post_plot, pos_post,     "Post (Remaining Pre Users)"),
        (G_rand_plot, pos_rand,     "Random Survival-Only CF"),
    ]

    for ax, (G, pos, title) in zip(axes, panels):

        nx.draw_networkx_nodes(
            G, pos,
            ax=ax,
            node_size=node_size,
            node_color="gray",
            alpha=node_alpha,
            linewidths=0
        )

        nx.draw_networkx_edges(
            G, pos,
            ax=ax,
            width=edge_widths(G),
            alpha=edge_alpha,
            edge_color="black"
        )

        ax.set_title(
            f"{title}\n|V|={G.number_of_nodes()}, |E|={G.number_of_edges()}",
            fontsize=13
        )

        ax.axis("off")

    plt.show()

In [ ]:
random_run_id = 1
cf_graph_path = os.path.join(survivor_dir, f"CF_graph_run{random_run_id:04d}_Oct2021_R11.pkl")
G_random = load_nx_graph(cf_graph_path)

plot_pre_post_random_simple(
    pre_graphs["Oct2021"],
    post_remaining_graphs["Jan2022"],
    G_random,
    weight="weight",
)

In [ ]:
random_run_id = 20
cf_graph_path = os.path.join(survivor_dir, f"CF_graph_run{random_run_id:04d}_Oct2021_R11.pkl")
G_random = load_nx_graph(cf_graph_path)

plot_pre_post_random_simple(
    pre_graphs["Oct2021"],
    post_remaining_graphs["Jan2022"],
    G_random,
    weight="weight",
)

In [ ]:
def plot_pre_post_random(
    G_pre,
    G_post,
    G_rand,
    weight="weight",
    seed=0,
    iterations=40,
    node_size=3,
    node_alpha=0.4,
    edge_alpha=0.08,
    base_edge_width=2.0,
    figsize=(18, 6),
):
    """
    Plot Pre, Post, and Random CF using a shared layout.
    Isolates removed for visualization only.
    Edge widths scaled globally across all graphs.
    """

    def remove_isolates(G):
        nodes = [n for n, d in G.degree() if d > 0]
        return G.subgraph(nodes).copy()

    G_pre  = remove_isolates(G_pre)
    G_post = remove_isolates(G_post)
    G_rand = remove_isolates(G_rand)

    G_union = nx.compose_all([G_pre, G_post, G_rand])
    pos = nx.spring_layout(G_union, seed=seed, iterations=iterations)


    all_weights = []

    for G in [G_pre, G_post, G_rand]:
        for _, _, d in G.edges(data=True):
            w = float(d.get(weight, 1.0))
            if w > 0:
                all_weights.append(w)

    global_max = max(all_weights) if all_weights else 1.0

    def edge_widths(G):
        widths = []
        for _, _, d in G.edges(data=True):
            w = float(d.get(weight, 1.0))
            w = max(w, 0.0)
            widths.append(base_edge_width * (w / global_max))
        return widths

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    panels = [
        (G_pre,  "Pre",  "skyblue"),
        (G_post, "Post (Remaining Pre Users)", "salmon"),
        (G_rand, "Random Survival CF", "gray"),
    ]

    for ax, (G, title, color) in zip(axes, panels):

        nx.draw_networkx_nodes(
            G, pos,
            ax=ax,
            node_size=node_size,
            node_color=color,
            alpha=node_alpha,
            linewidths=0
        )

        nx.draw_networkx_edges(
            G, pos,
            ax=ax,
            width=edge_widths(G),
            alpha=edge_alpha,
            edge_color="black"
        )

        ax.set_title(f"{title}\n|V|={G.number_of_nodes()}, |E|={G.number_of_edges()}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
random_run_id = 1

cf_path = os.path.join(
    survivor_dir,
    f"CF_graph_run{random_run_id:04d}_Oct2021_R11.pkl"
)

G_random = load_nx_graph(cf_path)

plot_pre_post_random(
    pre_graphs["Oct2021"],
    post_remaining_graphs["Jan2022"],
    G_random,
    weight="weight",
)

## Plotting with geography

In [ ]:
from shapely.geometry import LineString

In [ ]:
start_date = 20211001
end_date = 20211131

freq_home_path = os.path.join(home_dir, f"freq_home_{start_date}_{end_date}")
home_df = pd.read_csv(freq_home_path)

home_df = home_df.rename(columns={"cuebiq_id": "user", "fips_code": "pre_disaster_fips"})

# IMPORTANT: your graph nodes are int, so make home_df user int
home_df["user"] = pd.to_numeric(home_df["user"], errors="coerce").astype("Int64")
home_df = home_df.dropna(subset=["user"]).copy()
home_df["user"] = home_df["user"].astype(int)

home_df.head()

In [ ]:
home_df.columns

In [ ]:
def pick_latlon_cols(df):
    lat_cands = ["centroid_lat"]
    lon_cands = ["centroid_lng"]

    lat_col = next((c for c in lat_cands if c in df.columns), None)
    lon_col = next((c for c in lon_cands if c in df.columns), None)

    if lat_col is None or lon_col is None:
        raise KeyError(
            f"Could not find lat/lon columns in home_df.\n"
            f"Looked for lat in {lat_cands}\n"
            f"and lon in {lon_cands}\n"
            f"Available columns: {df.columns.tolist()}"
        )
    return lat_col, lon_col

lat_col, lon_col = pick_latlon_cols(home_df)
print("Using:", lat_col, lon_col)

In [ ]:
# choose these once
lat_col = "centroid_lat"
lon_col = "centroid_lng"

# force home_df user id type
home_df["user"] = home_df["user"].astype(str)

def nodes_to_home_gdf(G, home_df, lat_col, lon_col):
    # force graph node ids to same type as home_df["user"]
    nodes = pd.DataFrame({"user": [str(n) for n in G.nodes()]})

    dfm = nodes.merge(home_df[["user", lat_col, lon_col]], on="user", how="left")
    dfm = dfm.dropna(subset=[lat_col, lon_col]).copy()

    gdf = gpd.GeoDataFrame(
        dfm,
        geometry=gpd.points_from_xy(dfm[lon_col], dfm[lat_col]),
        crs="EPSG:4326"
    ).to_crs(epsg=3857)

    return gdf

g_pre  = nodes_to_home_gdf(G_pre,  home_df, lat_col, lon_col)
g_post = nodes_to_home_gdf(G_post, home_df, lat_col, lon_col)
g_rand = nodes_to_home_gdf(G_cf_norm, home_df, lat_col, lon_col)

In [ ]:
geo_path = os.path.join(base, "geography_CO", "cbg_colorado_geo.csv")
cbg = pd.read_csv(geo_path)

# geometry_wkt -> geometry
cbg["geometry"] = cbg["geometry_wkt"].apply(wkt.loads)
cbg_gdf = gpd.GeoDataFrame(cbg, geometry="geometry", crs="EPSG:4326")

# keep Boulder County CBGs (prefix 08013)
cbg_gdf["cbg_fips_str"] = cbg_gdf["fips_code"].astype(str).str.replace(".0", "", regex=False).str.zfill(12)
boulder_cbg = cbg_gdf[cbg_gdf["cbg_fips_str"].str.startswith("8013")].copy()

# project for plotting with your edges/nodes
boulder_cbg = boulder_cbg.to_crs(epsg=3857)


In [ ]:
run = 1
graph_path = os.path.join(
    boot_dir_w,
    f"CF_graph_run{run:04d}_Oct2021_{revision}.pkl"
)
G_cf2 = load_nx_graph(graph_path)
G_cf2_norm = normalize_random_to_post_rules(G_cf2)

print("Original CF2:", G_cf2.number_of_nodes(), G_cf2.number_of_edges())
print("Normalized CF2:", G_cf2_norm.number_of_nodes(), G_cf2_norm.number_of_edges())

# --- existing map code (unchanged) ---
lat_col = "centroid_lat"
lon_col = "centroid_lng"

home_df2 = home_df.copy()
home_df2["user"] = home_df2["user"].astype(str)
home_df2["pre_disaster_fips"] = (
    home_df2["pre_disaster_fips"].astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(12)
)

home_df2 = home_df2[home_df2["pre_disaster_fips"].str.startswith("08013")].copy()
coord_map = dict(zip(home_df2["user"], zip(home_df2[lon_col], home_df2[lat_col])))

def graph_edges_to_gdf(G, coord_map, *, graph_weight_attr="weight", top_k=None):
    rows = []
    for u, v, d in G.edges(data=True):
        u = str(u); v = str(v)
        if (u not in coord_map) or (v not in coord_map):
            continue

        w = d.get(graph_weight_attr, 1.0)
        try:
            w = float(w)
        except:
            w = 1.0

        (ux, uy) = coord_map[u]
        (vx, vy) = coord_map[v]

        rows.append({
            "u": u,
            "v": v,
            "w": w,
            "geometry": LineString([(ux, uy), (vx, vy)])
        })

    edges_gdf = gpd.GeoDataFrame(rows, crs="EPSG:4326")
    if edges_gdf.empty:
        return edges_gdf

    if top_k is not None:
        edges_gdf = edges_gdf.sort_values("w", ascending=False).head(int(top_k)).copy()

    return edges_gdf.to_crs(epsg=3857)

def graph_nodes_to_gdf(G, coord_map):
    nodes = []
    for n in G.nodes():
        s = str(n)
        if s in coord_map:
            x, y = coord_map[s]
            nodes.append({"user": s, "geometry": gpd.points_from_xy([x], [y])[0]})
    gdf = gpd.GeoDataFrame(nodes, crs="EPSG:4326")
    return gdf.to_crs(epsg=3857)

cbg_path = os.path.join(base, "geography_CO", "cbg_colorado_geo.csv")
cbg = pd.read_csv(cbg_path)

# make geometry
cbg["geometry"] = cbg["geometry_wkt"].apply(lambda x: wkt.loads(x) if isinstance(x, str) else None)
cbg_gdf = gpd.GeoDataFrame(cbg, geometry="geometry", crs="EPSG:4326")

# clean GEOID (pick the column you actually have)
# common names: 'fips_code' or 'geography_id'
if "fips_code" in cbg_gdf.columns:
    cbg_gdf["cbg_fips"] = cbg_gdf["fips_code"].astype(str).str.replace(".0","", regex=False).str.zfill(12)
elif "geography_id" in cbg_gdf.columns:
    cbg_gdf["cbg_fips"] = cbg_gdf["geography_id"].astype(str).str.replace(".0","", regex=False).str.zfill(12)
else:
    raise ValueError("Can't find a CBG id column in cbg_colorado_geo.csv (expected fips_code or geography_id)")

# Boulder County = 08013
boulder_cbg = cbg_gdf[cbg_gdf["cbg_fips"].str.startswith("08013")].copy()

print("Boulder CBG rows:", len(boulder_cbg), "| empty?", boulder_cbg.empty)

# CRS MUST match your edges/nodes (you convert those to 3857)
boulder_cbg_3857 = boulder_cbg.to_crs(epsg=3857)





In [ ]:
def plot_graph_on_map(
    G,
    coord_map,
    *,
    cbg_layer=None,
    graph_weight_attr="weight",
    top_k_edges=None,
    node_size=4,
    edge_alpha=0.3,
    node_alpha=0.5,
    max_edge_width=4.0,
    figsize=(7, 7),
    title=None,
):
    edges = graph_edges_to_gdf(G, coord_map, graph_weight_attr=graph_weight_attr, top_k=top_k_edges)
    nodes = graph_nodes_to_gdf(G, coord_map)

    fig, ax = plt.subplots(figsize=figsize)

    if cbg_layer is not None and not cbg_layer.empty:
        cbg_layer.boundary.plot(ax=ax, linewidth=0.6, color="lightgrey", alpha=0.8, zorder=0)

    if not edges.empty:
        w = edges["w"].replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
        wlog = np.log1p(w)
        denom = wlog.max() if wlog.max() > 0 else 1.0
        edges["lw"] = max_edge_width * (wlog / denom)
        edges.plot(ax=ax, linewidth=edges["lw"], alpha=edge_alpha, color="black", zorder=1)

    if not nodes.empty:
        nodes.plot(ax=ax, markersize=node_size, alpha=node_alpha, color="grey", zorder=2)

    ax.set_axis_off()
    if title:
        ax.set_title("")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_graph_on_map(G_pre, coord_map, cbg_layer=boulder_cbg_3857, top_k_edges=None, title="Pre (Boulder-only)")
plot_graph_on_map(G_post, coord_map, cbg_layer=boulder_cbg_3857, top_k_edges=None, title="Post (Boulder-only)")
plot_graph_on_map(G_cf_norm, coord_map, cbg_layer=boulder_cbg_3857, top_k_edges=None, title="Random CF (Boulder-only)")
plot_graph_on_map(G_cf2_norm, coord_map, cbg_layer=boulder_cbg_3857, top_k_edges=None, title=f"Yhat CF run={run:04d} (Boulder-only)")

In [ ]:
# =========================================================
# 1. prepare home data exactly like before
# =========================================================
home_df2 = home_df.copy()
home_df2["user"] = home_df2["user"].astype(str)
home_df2["pre_disaster_fips"] = (
    home_df2["pre_disaster_fips"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(12)
)

home_df2 = home_df2[home_df2["pre_disaster_fips"].str.startswith("08013")].copy()
home_df2["tract_fips"] = home_df2["pre_disaster_fips"].str[:11]

# =========================================================
# 2. prepare polygon layer
# =========================================================
boulder_cbg_3857 = boulder_cbg_3857.copy()
boulder_cbg_3857["cbg_fips"] = (
    boulder_cbg_3857["cbg_fips"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(12)
)
boulder_cbg_3857["tract_fips"] = boulder_cbg_3857["cbg_fips"].str[:11]

# =========================================================
# 3. dissolve CBGs into tract polygons
# =========================================================
tract_gdf = boulder_cbg_3857.dissolve(by="tract_fips", as_index=False).copy()

tract_gdf["centroid"] = tract_gdf.geometry.centroid
tract_gdf["cx"] = tract_gdf["centroid"].x
tract_gdf["cy"] = tract_gdf["centroid"].y

# =========================================================
# 4. geographic center of the study area
# =========================================================
study_union = tract_gdf.geometry.unary_union
study_center = study_union.centroid

center_x = study_center.x
center_y = study_center.y

tract_gdf["dist_to_center"] = np.sqrt(
    (tract_gdf["cx"] - center_x) ** 3 +
    (tract_gdf["cy"] - center_y) ** 3
)

top10_tracts_geo = (
    tract_gdf.sort_values("dist_to_center", ascending=True)
    .head(10)["tract_fips"]
    .tolist()
)

print("Top 10 geographically central tracts:")
print(top10_tracts_geo)

# =========================================================
# 5. filter home_df2 to only those central tracts
# =========================================================
home_df2 = home_df2[home_df2["tract_fips"].isin(top10_tracts_geo)].copy()
coord_map = dict(zip(home_df2["user"], zip(home_df2[lon_col], home_df2[lat_col])))

print("Users in selected central tracts:", len(coord_map))

# =========================================================
# 6. filter polygons to those same tracts
# =========================================================
cbg_layer_local = boulder_cbg_3857[
    boulder_cbg_3857["tract_fips"].isin(top10_tracts_geo)
].copy()

print("CBGs in selected central tracts:", len(cbg_layer_local))

# =========================================================
# 7. plot exactly like before
# =========================================================
plot_graph_on_map(
    G_pre,
    coord_map,
    cbg_layer=cbg_layer_local,
    top_k_edges=None,
    title="Pre - Geographically Central Tracts"
)

plot_graph_on_map(
    G_post,
    coord_map,
    cbg_layer=cbg_layer_local,
    top_k_edges=None,
    title="Post - Geographically Central Tracts"
)

plot_graph_on_map(
    G_cf_norm,
    coord_map,
    cbg_layer=cbg_layer_local,
    top_k_edges=None,
    title="Random CF - Geographically Central Tracts"
)

plot_graph_on_map(
    G_cf2_norm,
    coord_map,
    cbg_layer=cbg_layer_local,
    top_k_edges=None,
    title="Yhat CF - Geographically Central Tracts"
)

# Plotting Edge Weights

In [ ]:
csv_path = os.path.join(survivor_dir, f"combined_node_centralities_{revision}.csv")

df = pd.read_csv(csv_path)
df.head()

In [ ]:
df["phase"].unique()

In [ ]:
# fig, ax = plt.subplots(figsize=(7, 6))

# phase_map = {
#     "pre": "Pre-Disaster",
#     "post": "Post-Disaster",
#     "random_post": "Random Survival CF"
# }

# phase_color = {
#     "pre": "skyblue",
#     "post": "salmon",
#     "random_post": "gray"
# }

# phase_marker = {
#     "pre": "o",
#     "post": "s",
# }

# # ---------------------------------------------------
# # 1️⃣ Define global bins (important!)
# # ---------------------------------------------------

# all_degrees = df[df["raw_degree"] > 0]["raw_degree"]

# bins = np.logspace(
#     np.log10(all_degrees.min()),
#     np.log10(all_degrees.max()),
#     25
# )

# # ---------------------------------------------------
# # 2️⃣ Pre & Post
# # ---------------------------------------------------

# for phase in ["pre", "post"]:

#     df_phase = df[df["phase"] == phase]
#     degrees = df_phase["raw_degree"]
#     degrees = degrees[degrees > 0]

#     counts, _ = np.histogram(degrees, bins=bins, density=True)
#     counts[counts == 0] = np.nan

#     ax.plot(
#         bins[:-1],
#         counts,
#         marker=phase_marker[phase],
#         color=phase_color[phase],
#         linewidth=2,
#         label=phase_map[phase]
#     )

# # ---------------------------------------------------
# # 3️⃣ Random — compute distribution per run
# # ---------------------------------------------------

# random_df = df[df["phase"] == "random_post"]

# run_distributions = []

# for run in random_df["run"].unique():

#     df_run = random_df[random_df["run"] == run]
#     degrees = df_run["raw_degree"]
#     degrees = degrees[degrees > 0]

#     counts, _ = np.histogram(degrees, bins=bins, density=True)
#     run_distributions.append(counts)

# run_distributions = np.array(run_distributions)

# mean_random = np.nanmean(run_distributions, axis=0)
# std_random = np.nanstd(run_distributions, axis=0)

# # Mean curve
# ax.plot(
#     bins[:-1],
#     mean_random,
#     color=phase_color["random_post"],
#     linewidth=2,
#     label=phase_map["random_post"]
# )

# # Confidence band (±1 std)
# ax.fill_between(
#     bins[:-1],
#     mean_random - std_random,
#     mean_random + std_random,
#     color=phase_color["random_post"],
#     alpha=0.25
# )

# # ---------------------------------------------------
# # Styling
# # ---------------------------------------------------

# ax.set_xscale("log")
# ax.set_yscale("log")

# ax.set_xlabel("Degree (k)", fontsize=13)
# ax.set_ylabel("P(k)", fontsize=13)

# ax.spines["top"].set_visible(False)
# ax.spines["right"].set_visible(False)

# ax.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))

phase_map = {
    "pre": "Pre-Disaster",
    "post": "Post-Disaster",
    "random_post": "Random Survival CF"
}

phase_color = {
    "pre": "skyblue",
    "post": "salmon",
    "random_post": "lightgreen"
}

phase_marker = {
    "pre": "o",
    "post": "s", "random_post": "D"
}

all_strength = df[df["strength"] > 0]["strength"]

bins = np.logspace(
    np.log10(all_strength.min()),
    np.log10(all_strength.max()),
    25
)

bin_centers = bins[:-1]

for phase in ["pre", "post"]:

    df_phase = df[df["phase"] == phase]
    strengths = df_phase["strength"]
    strengths = strengths[strengths > 0]

    counts, _ = np.histogram(strengths, bins=bins, density=True)
    counts[counts == 0] = np.nan

    ax.plot(
        bin_centers,
        counts,
        marker=phase_marker[phase],
        color=phase_color[phase],
        linewidth=2,
        label=phase_map[phase]
    )

    phase_mean = strengths.mean()
    if pd.notna(phase_mean) and phase_mean > 0:
        ax.axvline(
            phase_mean,
            color=phase_color[phase],
            linestyle="--",
            linewidth=1.5,
            alpha=0.9
        )

random_df = df[df["phase"] == "random_post"]

run_distributions = []

for run in random_df["run"].unique():

    df_run = random_df[random_df["run"] == run]
    strengths = df_run["strength"]
    strengths = strengths[strengths > 0]

    counts, _ = np.histogram(strengths, bins=bins, density=True)
    run_distributions.append(counts)

run_distributions = np.array(run_distributions)

mean_random = np.nanmean(run_distributions, axis=0)
std_random = np.nanstd(run_distributions, axis=0)

mean_random[mean_random == 0] = np.nan

ax.plot(
    bin_centers,
    mean_random,
    color=phase_color["random_post"],
    linewidth=2,
    label=phase_map["random_post"]
)

lower = np.maximum(mean_random - std_random, 1e-12)
upper = mean_random + std_random

ax.fill_between(
    bin_centers,
    lower,
    upper,
    color=phase_color["random_post"],
    alpha=0.25
)

random_mean_strength = random_df.loc[random_df["strength"] > 0, "strength"].mean()
if pd.notna(random_mean_strength) and random_mean_strength > 0:
    ax.axvline(
        random_mean_strength,
        color=phase_color["random_post"],
        linestyle="--",
        linewidth=1.5,
        alpha=0.9
    )

ax.set_xscale("log")
ax.set_yscale("log")

#ax.set_xlabel("Weighted Degree (Strength)", fontsize=13)
#ax.set_ylabel("P(k)", fontsize=14)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

#ax.legend()
plt.tight_layout()
plt.show()